# Columbus 311 Service Requests: 2025 Operations Analysis

**Question:** How many service requests did Columbus residents make in 2025, what were they about, and how fast did the city close them?

**Data:** City of Columbus 311 Service Requests (public ArcGIS REST service, `arc.columbus.gov`), every request reported between Jan 1 and Dec 31, 2025.

**Tools:** Python (pandas), SQL (SQLite), Matplotlib, Power BI

**Steps:** pull the data, profile data quality, clean and validate it, load it into SQL, answer 6 business questions, and export tables for a Power BI dashboard.

In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

raw = pd.read_csv('columbus_311_2025_raw.csv', dtype=str)
for c in ['REPORTED_DATE', 'STATUS_DATE']:
    raw[c] = pd.to_datetime(raw[c])
print(f'Raw rows: {len(raw):,}  |  Columns: {raw.shape[1]}')
raw.head(3)

Raw rows: 310,857  |  Columns: 11


              CASE_ID  STATUS       REPORTED_DATE         STATUS_DATE  \
0  CAS-2437503-Z5D4Z6  Closed 2025-01-09 13:22:21 2025-01-09 15:55:08   
1  CAS-2437743-K7N7X8  Closed 2025-01-09 15:53:35 2025-01-09 16:03:24   
2  CAS-2438042-X2K0F6  Closed 2025-01-09 18:28:19 2025-01-09 18:28:24   

  DEPARTMENT_NAME       DIVISION_NAME  \
0   Neighborhoods  311 Service Center   
1   Neighborhoods  311 Service Center   
2   Neighborhoods  311 Service Center   

                                    REQUEST_CATEGORY  \
0  Trash, Recycling, Yard Waste and Illegal Dumpi...   
1                                City Staff Requests   
2                                City Staff Requests   

                       REQUEST_TYPE  ZIP COLUMBUSCOMMUNITY COUNCILDISTRICT  
0    Bulk Related Information/Notes  NaN               NaN             NaN  
1         General Information/Notes  NaN               NaN             NaN  
2  Refuse Related Information/Notes  NaN               NaN             NaN  

## 1. Data quality profile

In [2]:
profile = pd.DataFrame({
    'missing': raw.isna().sum(),
    'pct_missing': (raw.isna().mean() * 100).round(1)
})
print(profile)
print()
print(f"Repeated CASE_IDs: {raw['CASE_ID'].duplicated().sum():,}")
print(f"True duplicates (same CASE_ID and REQUEST_TYPE): {raw.duplicated(['CASE_ID', 'REQUEST_TYPE']).sum():,}")
print()
print(raw['STATUS'].value_counts())

                   missing  pct_missing
CASE_ID                  0          0.0
STATUS                   0          0.0
REPORTED_DATE            0          0.0
STATUS_DATE           9533          3.1
DEPARTMENT_NAME        186          0.1
DIVISION_NAME          268          0.1
REQUEST_CATEGORY      1865          0.6
REQUEST_TYPE             0          0.0
ZIP                  50141         16.1
COLUMBUSCOMMUNITY    80902         26.0
COUNCILDISTRICT      81572         26.2



Repeated CASE_IDs: 13,672
True duplicates (same CASE_ID and REQUEST_TYPE): 955

STATUS
Closed                    287220
Canceled                   13620
Work Order In Progress      3095
In Progress                 2557
Received                    2459
Written Orders Issued       1367
Re-Assigned                  513
Closed | Closed               21
On Hold                        2
Returned                       2
Duplicate                      1
Name: count, dtype: int64


## 2. Cleaning rules

1. **Duplicates:** a single case can hold several request types, so a duplicate is the same `CASE_ID` *and* `REQUEST_TYPE`. Keep the most recent status update.
2. **Time zone:** the source stores timestamps in UTC; convert them to Eastern time so late-December requests land in the right month.
3. **Internal staff logs:** drop the *City Staff Requests* category and *Information/Notes* request types. They are staff notes, not resident requests, and close in seconds, which would skew timing.
4. **Missing labels:** fill blank department, division, category, district, community, and ZIP with `Unknown`.
5. **Timing:** `days_to_close` is calculated only for *Closed* requests that have a close date. Closed requests with no close date are flagged and kept out of timing. Canceled and still-open requests are kept for volume counts but excluded from timing.

In [3]:
df = raw.sort_values('STATUS_DATE').drop_duplicates(['CASE_ID', 'REQUEST_TYPE'], keep='last').copy()
for c in ['REPORTED_DATE', 'STATUS_DATE']:
    df[c] = df[c].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None)
dupes_removed = len(raw) - len(df)

internal = (df['REQUEST_CATEGORY'] == 'City Staff Requests') | df['REQUEST_TYPE'].str.contains('Information/Notes', na=False)
internal_removed = int(internal.sum())
df = df[~internal].copy()

for c in ['DEPARTMENT_NAME', 'DIVISION_NAME', 'REQUEST_CATEGORY', 'COUNCILDISTRICT', 'COLUMBUSCOMMUNITY', 'ZIP']:
    df[c] = df[c].fillna('Unknown').replace('', 'Unknown')

df['STATUS'] = df['STATUS'].replace({'Closed | Closed': 'Closed'})
df['status_group'] = df['STATUS'].map(lambda s: 'Closed' if s == 'Closed' else ('Canceled' if s == 'Canceled' else 'Open'))
missing_close = (df['status_group'] == 'Closed') & df['STATUS_DATE'].isna()
closed = (df['status_group'] == 'Closed') & df['STATUS_DATE'].notna()
df['days_to_close'] = ((df['STATUS_DATE'] - df['REPORTED_DATE']).dt.total_seconds() / 86400).where(closed).round(2)
df['closed_within_7_days'] = (df['days_to_close'] <= 7).where(closed)
df['report_month'] = df['REPORTED_DATE'].dt.to_period('M').astype(str)

print(f'Duplicates removed:     {dupes_removed:,}')
print(f'Internal logs removed:  {internal_removed:,}')
print(f'Closed with no close date (flagged, excluded from timing): {int(missing_close.sum()):,}')
print(f'Clean resident requests: {len(df):,}')

Duplicates removed:     955
Internal logs removed:  100,325
Closed with no close date (flagged, excluded from timing): 6,451
Clean resident requests: 209,577


## 3. Validation checks

In [4]:
checks = {
    'No duplicate CASE_ID + REQUEST_TYPE pairs': not df.duplicated(['CASE_ID', 'REQUEST_TYPE']).any(),
    'Row counts reconcile (raw - duplicates - internal = clean)': len(raw) - dupes_removed - internal_removed == len(df),
    'No closed request closed before it was reported': (df.loc[closed, 'days_to_close'] >= 0).all(),
    'Every timed request has a close date': df.loc[df['days_to_close'].notna(), 'STATUS_DATE'].notna().all(),
    'All requests reported in 2025 (Eastern time)': df['REPORTED_DATE'].dt.year.eq(2025).all(),
}
for name, ok in checks.items():
    print(('PASS  ' if ok else 'FAIL  ') + name)
print(f"\n{sum(checks.values())}/{len(checks)} checks passed")

PASS  No duplicate CASE_ID + REQUEST_TYPE pairs
PASS  Row counts reconcile (raw - duplicates - internal = clean)
PASS  No closed request closed before it was reported
PASS  Every timed request has a close date
PASS  All requests reported in 2025 (Eastern time)

5/5 checks passed


## 4. Load into SQL

A small star schema: one fact table of requests and one dimension table of request types, joined on `request_type`.

In [5]:
con = sqlite3.connect(':memory:')
fact = df[['CASE_ID', 'REPORTED_DATE', 'report_month', 'REQUEST_TYPE', 'status_group', 'days_to_close',
           'closed_within_7_days', 'COUNCILDISTRICT', 'COLUMBUSCOMMUNITY', 'ZIP']].rename(columns=str.lower)
dim = (df[['REQUEST_TYPE', 'REQUEST_CATEGORY', 'DEPARTMENT_NAME', 'DIVISION_NAME']]
       .drop_duplicates('REQUEST_TYPE').rename(columns=str.lower))
fact.to_sql('requests', con, index=False)
dim.to_sql('request_types', con, index=False)
q = lambda s: pd.read_sql(s, con)
print(f"requests: {len(fact):,} rows  |  request_types: {len(dim):,} rows")

requests: 209,577 rows  |  request_types: 468 rows


### Q1. Which categories get the most requests?

In [6]:
q1 = q('''
SELECT t.request_category,
       COUNT(*) AS requests,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM requests), 1) AS pct_of_total
FROM requests r
JOIN request_types t ON r.request_type = t.request_type
GROUP BY t.request_category
ORDER BY requests DESC;
''')
q1

                                     request_category  requests  pct_of_total
0   Trash, Recycling, Yard Waste and Illegal Dumpi...    102594          49.0
1   Streets, Sidewalks, Street Lighting, Sign and ...     34416          16.4
2   City Code Violations, Noise, Building and Priv...     23949          11.4
3        Public Safety and Traffic Enforcement Issues     17054           8.1
4   Park, Bike Path, Recreation Center, Pool and T...      7322           3.5
5   Water, Drinking Water, Storm Water, Flooding, ...      7181           3.4
6             Animal, Pets, Pests and Wildlife Issues      5094           2.4
7                       Public Comments and Questions      3503           1.7
8                                 Homelessness Issues      2398           1.1
9                                             Unknown      1866           0.9
10                           Bikes and Scooter Issues      1009           0.5
11                                        Electricity       957 

### Q2. What are the top 10 request types?

In [7]:
q2 = q('''
SELECT request_type, COUNT(*) AS requests
FROM requests
GROUP BY request_type
ORDER BY requests DESC
LIMIT 10;
''')
q2

                               request_type  requests
0        90-Gallon Container Needs Serviced     15992
1                 Repair Of Street Lighting      9827
2        Replace 90-Gallon Refuse Container      9730
3      Code/Weeds or High Grass on Property      7833
4        Abandoned Vehicle On Street/R.O.W.      7090
5            Illegal Materials in the Alley      6808
6             Code/Trash and Debris in Yard      6647
7  Bulk Manual Service/Waiting on Container      6291
8                Recycling - Missed Service      5265
9                    Pothole On City Street      4375

### Q3. How fast does each department close requests?

In [8]:
q3 = q('''
SELECT t.department_name,
       COUNT(*) AS closed_requests,
       ROUND(AVG(r.days_to_close), 1) AS avg_days_to_close,
       ROUND(100.0 * AVG(r.closed_within_7_days), 1) AS pct_closed_within_7_days
FROM requests r
JOIN request_types t ON r.request_type = t.request_type
WHERE r.status_group = 'Closed'
GROUP BY t.department_name
HAVING COUNT(*) >= 500
ORDER BY avg_days_to_close DESC;
''')
q3

        department_name  closed_requests  avg_days_to_close  \
0   Building and Zoning            27571               25.2   
1                Health             2961               22.8   
2  Recreation and Parks             4826               21.3   
3         Public Safety            18436               18.5   
4        Public Service           112625               10.9   
5      Public Utilities            14302                8.2   
6         Neighborhoods             5449                4.9   

   pct_closed_within_7_days  
0                      44.6  
1                      46.4  
2                      60.5  
3                      51.5  
4                      64.6  
5                      75.5  
6                      89.3  

### Q4. Which high-volume request types take the longest?

In [9]:
q4 = q('''
SELECT request_type,
       COUNT(*) AS closed_requests,
       ROUND(AVG(days_to_close), 1) AS avg_days_to_close
FROM requests
WHERE status_group = 'Closed'
GROUP BY request_type
HAVING COUNT(*) >= 1000
ORDER BY avg_days_to_close DESC
LIMIT 10;
''')
q4

                                   request_type  closed_requests  \
0                               SWI/Refuse/Misc             1087   
1                               Report a Dumper             1459   
2                Illegal Materials in the Alley             6411   
3                 Illegal Materials at the Curb             3562   
4       7A To 3P Parking Violations/City Street             2523   
5                             Code/Housing/Misc             2790   
6            Container Left At Collection Point             2144   
7               Trash/Debris along Right-Of-Way             1933   
8  Code/Inoperable or Junk Vehicles on Property             1333   
9             Code/Vehicle Parked on Grass/Lawn             1245   

   avg_days_to_close  
0               51.7  
1               33.8  
2               33.6  
3               32.0  
4               31.9  
5               31.6  
6               31.5  
7               30.0  
8               27.6  
9               25.8 

### Q5. How did request volume change month to month?

In [10]:
q5 = q('''
WITH monthly AS (
    SELECT report_month, COUNT(*) AS requests
    FROM requests
    GROUP BY report_month
)
SELECT report_month,
       requests,
       ROUND(100.0 * (requests - LAG(requests) OVER (ORDER BY report_month))
             / LAG(requests) OVER (ORDER BY report_month), 1) AS pct_change
FROM monthly
ORDER BY report_month;
''')
q5

   report_month  requests  pct_change
0       2025-01     14920         NaN
1       2025-02     13761        -7.8
2       2025-03     16915        22.9
3       2025-04     17654         4.4
4       2025-05     18404         4.2
5       2025-06     19985         8.6
6       2025-07     21714         8.7
7       2025-08     19485       -10.3
8       2025-09     21118         8.4
9       2025-10     17525       -17.0
10      2025-11     13171       -24.8
11      2025-12     14925        13.3

### Q6. Which council districts generate the most requests, and how fast are they closed?

In [11]:
q6 = q('''
SELECT councildistrict,
       COUNT(*) AS requests,
       ROUND(AVG(CASE WHEN status_group = 'Closed' THEN days_to_close END), 1) AS avg_days_to_close
FROM requests
WHERE councildistrict <> 'Unknown'
GROUP BY councildistrict
ORDER BY requests DESC;
''')
q6

  councildistrict  requests  avg_days_to_close
0      District 7     48846               14.8
1      District 4     30655               14.6
2      District 6     30082               16.1
3      District 8     29554               14.1
4      District 3     17948               13.0
5      District 5     13635               11.7
6      District 9     12344               14.9
7      District 2     11722               11.0
8      District 1     11632               12.0

## 5. Visualizations

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(q5['report_month'].str[5:], q5['requests'], marker='o', color='#1f5fae')
axes[0].set_title('Monthly 311 Requests, 2025')
axes[0].set_xlabel('Month'); axes[0].set_ylabel('Requests'); axes[0].grid(alpha=0.3)
top = q2.iloc[::-1]
axes[1].barh(top['request_type'], top['requests'], color='#1f5fae')
axes[1].set_title('Top 10 Request Types'); axes[1].set_xlabel('Requests')
plt.tight_layout(); plt.savefig('311_overview.png', dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
d = q3.sort_values('avg_days_to_close')
ax.barh(d['department_name'], d['avg_days_to_close'], color='#bb0000')
ax.set_title('Average Days to Close by Department'); ax.set_xlabel('Days')
plt.tight_layout(); plt.savefig('311_days_to_close.png', dpi=150); plt.show()

## 6. Export tables for Power BI

In [13]:
fact.to_csv('powerbi_fact_requests.csv', index=False)
dim.to_csv('powerbi_dim_request_types.csv', index=False)
q5.to_csv('summary_monthly.csv', index=False)
q1.to_csv('summary_categories.csv', index=False)
q3.to_csv('summary_departments.csv', index=False)
print('Exported: powerbi_fact_requests.csv, powerbi_dim_request_types.csv, and 3 summary tables')

Exported: powerbi_fact_requests.csv, powerbi_dim_request_types.csv, and 3 summary tables


## 7. Key findings

- **310,857 raw records → 209,577 resident requests** after removing 955 true duplicates and 100,325 internal staff logs. All 5 validation checks passed.
- **Not every repeated ID is a duplicate.** 13,672 case IDs repeat, but most are one case holding several request types. Deduplicating on case ID alone would have deleted about 12,700 real requests.
- **Time zone fix:** timestamps come in UTC. Converting to Eastern moved late-December requests out of "January 2026" and back into 2025.
- **6,451 requests were marked Closed with no close date.** They are flagged and kept out of timing metrics.
- **Trash, recycling, and yard waste make up 49% of all requests.** The top single request type is "90-Gallon Container Needs Serviced" (15,992).
- **Volume peaked in July (21,714)** and dropped **24.8%** from October to November.
- **Citywide, requests closed in 14.0 days on average (median 4.2), and 61.4% closed within 7 days.**
- **Building and Zoning was slowest** (25.2 days on average, 44.6% within a week). **Neighborhoods was fastest** (4.9 days, 89.3% within a week).
- **District 7 generated the most requests** (48,846), more than any other council district.

**Data source:** City of Columbus 311 Service Requests, ArcGIS REST service `arc.columbus.gov/arcgis/rest/services/Applications/ServiceRequests/MapServer/1`, queried for `REPORTED_DATE` in 2025 (paged 2,000 records per call).